In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [5]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/TicketMind/data/train_data.csv")
test_df = pd.read_csv("/content/drive/MyDrive/TicketMind/data/test_data.csv")

train_df.shape, test_df.shape

((19708, 6), (4927, 6))

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["intent"])
test_df["label"] = le.transform(test_df["intent"])

num_labels = len(le.classes_)
print(f"categories numbers: {num_labels}")
print(le.classes_)

categories numbers: 27
['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_invoice'
 'get_refund' 'newsletter_subscription' 'payment_issue' 'place_order'
 'recover_password' 'registration_problems' 'review'
 'set_up_shipping_address' 'switch_account' 'track_order' 'track_refund']


In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["instruction", "label"]])
test_dataset = Dataset.from_pandas(test_df[["instruction", "label"]])

train_dataset, test_dataset

(Dataset({
     features: ['instruction', 'label'],
     num_rows: 19708
 }),
 Dataset({
     features: ['instruction', 'label'],
     num_rows: 4927
 }))

In [8]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["instruction"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/19708 [00:00<?, ? examples/s]

Map:   0%|          | 0/4927 [00:00<?, ? examples/s]

In [10]:
train_tokenized[0]

{'instruction': 'i have a question about  canceling order {{Order Number}}',
 'label': 0,
 'input_ids': [101,
  1045,
  2031,
  1037,
  3160,
  2055,
  17542,
  2075,
  2344,
  1063,
  1063,
  2344,
  2193,
  1065,
  1065,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/TicketMind/model_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="/content/drive/MyDrive/TicketMind/logs",
    logging_steps=50,
    report_to="none",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.030381,0.021007,0.995535,0.994731
2,0.007132,0.015317,0.996753,0.996486
3,0.005287,0.012728,0.997159,0.996877


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3696, training_loss=0.18284886901657948, metrics={'train_runtime': 247.377, 'train_samples_per_second': 239.004, 'train_steps_per_second': 14.941, 'total_flos': 489718393392384.0, 'train_loss': 0.18284886901657948, 'epoch': 3.0})

In [19]:
sample = test_df.iloc[0]
print("message:", sample["instruction"])
print("intent:", sample["intent"])

message: I have tl talk to a live agent
intent: contact_human_agent


In [21]:
import torch
import torch.nn.functional as F

def predict_intent(text: str, model, tokenizer, label_encoder, device="cuda"):
    model.eval()
    model.to(device)

    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)

    predicted_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][predicted_id].item()

    predicted_intent = label_encoder.inverse_transform([predicted_id])[0]

    return {
        "intent": predicted_intent,
        "confidence": round(confidence, 4),
    }

In [23]:
result = predict_intent(
    "I want to cancel my order please, it's taking too long",
    model,
    tokenizer,
    le,
)
print(result)

{'intent': 'cancel_order', 'confidence': 0.9959}


In [28]:
test_messages = [
    "how can I get a refund for my purchase?",
    "I need to talk to a real person, not a bot",
    "can you change my shipping address?",
]

for msg in test_messages:
    result = predict_intent(msg, model, tokenizer, le)
    print(f"message: {msg}")
    print(f"  → intent: {result['intent']} (confidence: {result['confidence']*100:.1f}%)\n")

message: how can I get a refund for my purchase?
  → intent: get_refund (confidence: 99.9%)

message: I need to talk to a real person, not a bot
  → intent: contact_human_agent (confidence: 99.9%)

message: can you change my shipping address?
  → intent: change_shipping_address (confidence: 99.9%)



In [16]:
final_model_path = "/content/drive/MyDrive/TicketMind/model_final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/TicketMind/model_final/tokenizer_config.json',
 '/content/drive/MyDrive/TicketMind/model_final/tokenizer.json')

In [17]:
import pickle

with open(final_model_path + "/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("saved")

saved
